In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout_rate=0.0):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.dropout = nn.Dropout2d(dropout_rate) if dropout_rate > 0 else nn.Identity()

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.dropout(out)
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return F.relu(out)

# --- NÂNG CẤP: Thêm cơ chế Self-Attention ---
class SimpleAttention(nn.Module):
    def __init__(self, channel):
        super(SimpleAttention, self).__init__()
        self.query = nn.Conv2d(channel, channel // 8, 1)
        self.key = nn.Conv2d(channel, channel // 8, 1)
        self.value = nn.Conv2d(channel, channel, 1)
        self.gamma = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        batch_size, C, width, height = x.size()

        proj_query = self.query(x).view(batch_size, -1, width * height).permute(0, 2, 1)
        proj_key = self.key(x).view(batch_size, -1, width * height)

        energy = torch.bmm(proj_query, proj_key)
        attention = F.softmax(energy, dim=-1)

        proj_value = self.value(x).view(batch_size, -1, width * height)
        out = torch.bmm(proj_value, attention.permute(0, 2, 1))
        out = out.view(batch_size, C, width, height)

        out = self.gamma * out + x
        return out

class SquareCRNN(nn.Module):
    def __init__(self, num_classes, hidden_size=256, dropout_rate=0.3): # Tăng nhẹ dropout
        super(SquareCRNN, self).__init__()

        # 1. Trích xuất đặc trưng ban đầu
        self.conv1 = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        # 2. Khối Residual
        self.layer1 = ResidualBlock(64, 128, stride=1, dropout_rate=0.1)
        self.maxpool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.layer2 = ResidualBlock(128, 256, stride=1, dropout_rate=dropout_rate)
        self.maxpool3 = nn.MaxPool2d(kernel_size=(2, 2), stride=(2, 1), padding=(0, 1))

        self.layer3 = ResidualBlock(256, 512, stride=1, dropout_rate=dropout_rate)
        self.maxpool4 = nn.MaxPool2d(kernel_size=(2, 2), stride=(2, 1), padding=(0, 1))

        self.layer4 = ResidualBlock(512, 512, stride=1, dropout_rate=dropout_rate)

        # --- NÂNG CẤP: Gắn Attention trước khi ép kích thước ---
        self.attention = SimpleAttention(512)

        # 3. Ép chiều cao về 1
        self.adaptive_pool = nn.AdaptiveAvgPool2d((1, None))

        # 4. Sequence Modeling (RNN)
        self.rnn = nn.LSTM(512, hidden_size, bidirectional=True, num_layers=2, dropout=0.4) # Tăng dropout RNN để chống nhiễu chuỗi

        # 5. Phân loại
        self.fc = nn.Linear(hidden_size * 2, num_classes + 1)

        # Khởi tạo trọng số thông minh
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool1(x)

        x = self.layer1(x)
        x = self.maxpool2(x)

        x = self.layer2(x)
        x = self.maxpool3(x)

        x = self.layer3(x)
        x = self.maxpool4(x)

        x = self.layer4(x)

        # Gắn Attention giúp mô hình tìm ra các nét chữ quan trọng nhất
        x = self.attention(x)

        x = self.adaptive_pool(x)

        b, c, h, w = x.size()
        x = x.view(b, c * h, w)
        x = x.permute(2, 0, 1)

        x, _ = self.rnn(x)
        x = self.fc(x)
        return x

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import cv2
import numpy as np
import os
import random


class OCRDataset(Dataset):
    def __init__(self, label_file, char_map, img_dir, size=128, is_train=True):
        self.size = size
        self.img_dir = img_dir
        self.char_map = char_map
        self.is_train = is_train
        self.valid_data = []

        print("Đang rà soát và làm sạch dữ liệu...")
        missing_imgs, invalid_labels = 0, 0

        with open(label_file, 'r', encoding='utf-8-sig') as f:
            for line in f:
                parts = line.strip().split(',')
                if len(parts) >= 2:
                    img_name = parts[0]
                    raw_label = ','.join(parts[1:])
                    img_path = os.path.join(self.img_dir, img_name)

                    if not os.path.exists(img_path):
                        missing_imgs += 1
                        continue

                    clean_label = [self.char_map[c] for c in raw_label.upper() if c in self.char_map]

                    if len(clean_label) == 0:
                        invalid_labels += 1
                        continue

                    self.valid_data.append((img_path, clean_label))

        print(f"-> Loại bỏ: {missing_imgs} ảnh rỗng, {invalid_labels} nhãn sai.")
        print(f"-> Sẵn sàng huấn luyện: {len(self.valid_data)} mẫu.")

    def augment_image(self, img):
        """NÂNG CẤP: Data Augmentation "lì đòn" với thực tế"""
        # 1. Chỉnh sáng/tối
        alpha = random.uniform(0.7, 1.3)
        beta = random.randint(-30, 30)
        img = cv2.convertScaleAbs(img, alpha=alpha, beta=beta)

        # 2. Xoay nhẹ ảnh (Mô phỏng camera lắp bị nghiêng)
        if random.random() > 0.5:
            rows, cols = img.shape
            M = cv2.getRotationMatrix2D((cols/2, rows/2), random.uniform(-5, 5), 1)
            img = cv2.warpAffine(img, M, (cols, rows), borderValue=(0)) # Điền nền đen cho góc bị thiếu

        # 3. Thêm nhiễu Gaussian Noise (Mô phỏng camera dỏm, thiếu sáng)
        if random.random() > 0.7:
            gauss = np.random.normal(0, 15, img.size).reshape(img.shape).astype('uint8')
            img = cv2.add(img, gauss)

        return img

    def __len__(self):
        return len(self.valid_data)

    def __getitem__(self, idx):
        img_path, target = self.valid_data[idx]
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

        if img is None:
            img = np.zeros((self.size, self.size), dtype=np.uint8)
            target = [1]

        if self.is_train:
            img = self.augment_image(img)

        img = cv2.resize(img, (self.size, self.size))
        img = (img.astype(np.float32) / 127.5) - 1.0
        img = torch.from_numpy(img).unsqueeze(0)

        return img, torch.tensor(target, dtype=torch.long), len(target)

def collate_fn(batch):
    imgs, targets, target_lengths = zip(*batch)
    imgs = torch.stack(imgs)
    targets = torch.cat(targets)
    target_lengths = torch.tensor(target_lengths, dtype=torch.long)
    return imgs, targets, target_lengths

def main():
    chars = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ-."
    char_map = {c: i + 1 for i, c in enumerate(chars)}

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = SquareCRNN(len(chars)).to(device)

    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=4, factor=0.5, min_lr=1e-6)
    criterion = nn.CTCLoss(blank=0, zero_infinity=True).to(device)

    drive_dir = '/content/drive/MyDrive/AIOT'
    csv_path = f'{drive_dir}/train.csv'
    img_dir = f'{drive_dir}/images'

    if not os.path.exists(csv_path):
        print(f"Lỗi: Không tìm thấy {csv_path}. Bạn đã Mount Drive chưa?")
        return

    dataset = OCRDataset(csv_path, char_map, img_dir, is_train=True)

    if len(dataset) == 0:
        return

    loader = DataLoader(dataset, batch_size=32, shuffle=True, collate_fn=collate_fn, num_workers=2)

    print(f"--- Bắt đầu huấn luyện Siêu Mô Hình trên: {device} ---")

    epochs = 150
    best_loss = float('inf')

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0

        for imgs, targets, target_lengths in loader:
            imgs, targets, target_lengths = imgs.to(device), targets.to(device), target_lengths.to(device)

            optimizer.zero_grad()
            preds = model(imgs)

            input_lengths = torch.full(size=(imgs.size(0),), fill_value=preds.size(0), dtype=torch.long, device=device)

            loss = criterion(preds.log_softmax(2), targets, input_lengths, target_lengths)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)

            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(loader)
        scheduler.step(avg_loss)

        print(f"Epoch [{epoch}/{epochs}] - Loss: {avg_loss:.4f} - LR: {optimizer.param_groups[0]['lr']:.6f}")

        if avg_loss < best_loss:
            best_loss = avg_loss
            save_path = f'{drive_dir}/best_square_ocr_pro.pth'
            torch.save(model.state_dict(), save_path)
            print(f"  🔥 Đã lưu Mô hình nâng cấp vào: {save_path} (Loss Tốt Nhất: {best_loss:.4f})")

if __name__ == "__main__":
    main()